# Part 1 : Practise

## Task 1 : Load the loans and look for gaps

In [1]:
import pandas as pd

loans_raw = pd.read_csv("data/library_loans_clean.csv", parse_dates=["loan_date", "return_date"])
loans = loans_raw.copy()

In [2]:
loans.shape

(600, 8)

In [3]:
loans.dtypes

loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object

In [4]:
loans.isna().sum()

loan_id         0
member_id       0
title           0
genre           0
loan_date       3
return_date    36
late_fee        0
returned        0
dtype: int64

In [5]:
(loans.isna().mean()*100).round(1)

loan_id        0.0
member_id      0.0
title          0.0
genre          0.0
loan_date      0.5
return_date    6.0
late_fee       0.0
returned       0.0
dtype: float64

**Answer:** The table has 600 rows and 8 columns. The two date columns became dates because I used parse_dates.
Only two columns have missing values: loan_date has 3 missing (0.5 %) and return_date has 36 missing (6.0 %).

## Task 2 : Two Kinds of missing

In [6]:
loans[loans["loan_date"].isna()]

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
462,L-30463,M0007,Blue Moon Protocol,Sci-fi,NaT,2026-05-24,0.0,True
467,L-30468,M0018,Blue Moon Protocol,Sci-fi,NaT,2026-06-08,0.0,True
571,L-30572,M0090,Mimmi and the Moon,Children,NaT,2026-06-26,0.0,True


In [7]:
loans[loans["return_date"].isna()]

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
52,L-30053,M0064,The Brave Little Boat,Children,2026-01-21,NaT,0.0,False
68,L-30069,M0068,Shadow of the Tower,Fantasy,2026-01-25,NaT,0.0,False
90,L-30091,M0029,A Quiet Witness,Crime,2026-02-01,NaT,0.0,False
91,L-30092,M0144,Night Train to Nowhere,Crime,2026-02-01,NaT,0.0,False
93,L-30094,M0126,Wolves of the Old Wood,Fantasy,2026-01-31,NaT,0.0,False
100,L-30101,M0026,The Silent Harbour,Crime,2026-02-03,NaT,0.0,False
104,L-30105,M0110,Between the Lines,Romance,2026-02-05,NaT,0.0,False
110,L-30111,M0101,Between the Lines,Romance,2026-02-05,NaT,0.0,False
128,L-30129,M0027,Salt and Iron,History,2026-02-11,NaT,0.0,False
139,L-30140,M0119,Ollie Learns to Swim,Children,2026-02-15,NaT,0.0,False


In [8]:
loans[loans["return_date"].isna()].count()

loan_id        36
member_id      36
title          36
genre          36
loan_date      36
return_date     0
late_fee       36
returned       36
dtype: int64

In [9]:
loans[loans["return_date"].isna()]["returned"].value_counts()

returned
False    30
True      6
Name: count, dtype: int64

In [10]:
loans["has_return_date"] = loans["return_date"].notna()

In [11]:
loans["has_return_date"].value_counts()

has_return_date
True     564
False     36
Name: count, dtype: int64

**Answer:** 3 loans have no loan_date. 36 loans have no return_date. 30 of them are not returned yet,
so they are missing for a reason. 6 of them are returned but have no return date, so someone forgot
to write it. That is missing by accident. The 3 missing loan dates are also by accident, because
every loan must start on some day. has_return_date gives True 564 and False 36.

## Task 3 : Look at the members

In [12]:
members_raw = pd.read_csv("data/library_members.csv")
members = members_raw.copy()
members.shape

(148, 3)

In [13]:
members.info()

<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB


In [14]:
members["birth_year"].isna().sum()

np.int64(18)

In [15]:
members["birth_year"].describe()

count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64

In [16]:
members["branch"].describe()

count         136
unique          5
top       Central
freq           40
Name: branch, dtype: object

In [17]:
(members["birth_year"] == 1900).sum()

np.int64(8)

In [18]:
(members["birth_year"] == 2030).sum()

np.int64(3)

In [19]:
# duplicates in the whole row
members.duplicated().sum()

np.int64(4)

In [20]:
# duplicates in the member_id
members.duplicated(subset="member_id").sum()

np.int64(4)

**Answer:** The members table has 148 rows and 3 columns. I found these problems:
- branch has 12 missing values
- birth_year has 18 missing values
- 8 members have birth year 1900, which is too old, so I think it means "unknown"
- 3 members have birth year 2030, which is in the future, so it is wrong
- 4 rows are duplicated, and 4 member_id are duplicated, so the same members are in the table twice

## Task 4 : Placeholders and impossible values

In [21]:
members["birth_year"] = members["birth_year"].where(members["birth_year"].between(1920,2015))

In [22]:
members["birth_year"].isna().sum()

np.int64(29)

In [23]:
members["birth_year"].count()

np.int64(119)

In [24]:
members["birth_year"].describe()

count     119.000000
mean     1987.815126
std        12.062666
min      1957.000000
25%      1979.000000
50%      1988.000000
75%      1996.500000
max      2010.000000
Name: birth_year, dtype: float64

**Answer:** I changed the birth years that are not between 1920 and 2015 to missing values.
Now 29 birth years are missing (18 + 8 + 3). After this the count is 119, the mean is 1987.82,
min is 1957 and max is 2010. Before, the mean was 1983.38. The mean went up because the 1900
values were pulling it down.

## Task 5 : Remove, fill, or leave

### (a) How many rows are left with dropna (just looking, not saving)

In [25]:
print(members.dropna().shape)
print(members.dropna(subset=["branch"]).shape)
print(members.dropna(subset=["birth_year"]).shape)

(107, 3)
(136, 3)
(119, 3)


### (b) Put "Unknown" where branch is missing

In [27]:
members["branch"] = members["branch"].fillna("Unknown")
members["branch"].value_counts()

branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64

### (c) Try filling birth_year with the median (in a new variable)

In [28]:
median_year = members["birth_year"].median()
filled = members["birth_year"].fillna(median_year)

print("median:", median_year)
print("before: mean", round(members["birth_year"].mean(), 2), " std", round(members["birth_year"].std(), 2))
print("after:  mean", round(filled.mean(), 2), " std", round(filled.std(), 2))

median: 1988.0
before: mean 1987.82  std 12.06
after:  mean 1987.85  std 10.81


**Answer:** With dropna() only 107 rows are left. With subset branch 136 are left, and with subset birth_year 119.
If I remove every row with a gap I lose 41 members, so I don't do that.
After filling branch with "Unknown" I get Central 40, South 34, North 32, East 17, Harbour 13 and Unknown 12.
The median birth year is 1988. When I fill the missing years with it, the mean is almost the same (1987.82 and 1987.85),
but the std goes down from 12.06 to 10.81. That is because 29 people suddenly get the same year, so the ages look closer than they are.
So I keep birth_year missing and don't fill it.

## Task 6 : Merge, and check

### Shape of both tables

In [29]:
print(loans.shape)
print(members.shape)

(600, 9)
(148, 3)


### Merge loans and members (left, without validate)

In [30]:
merged = loans.merge(members, on="member_id", how="left")
merged.shape

(616, 11)

### late_fee total before and after

In [31]:
print("before:", loans["late_fee"].sum())
print("after:", merged["late_fee"].sum())

before: 7487.5
after: 7725.0


### Merge again with validate (this should give an error)

In [33]:
merged = loans.merge(members, on="member_id", how="left", validate="many_to_one")

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 member_id
    M0015
    M0033
    M0065
    M0071 ...

**Answer:** loans is (600, 9) and members is (148, 3).
The merge gives (616, 11), so 16 rows too many. 4 members are in the members table two times, so their loans got copied.
The late_fee total goes from 7487.5 to 7725.0, so 237.5 kr is counted twice.
With validate="many_to_one" I get MergeError: Merge keys are not unique in right dataset.
Right dataset is members. So validate shows the problem before I get wrong numbers.

## Task 7 : Fix the key, and find who is missing

### Remove the duplicate members

In [34]:
members = members.drop_duplicates().reset_index(drop=True)
members.shape

(144, 3)

### Merge again with validate and indicator

In [35]:
merged = loans.merge(members, on="member_id", how="left", validate="many_to_one", indicator=True)
print(merged.shape)
print("late_fee total:", merged["late_fee"].sum())
merged["_merge"].value_counts()

(600, 12)
late_fee total: 7487.5


_merge
both          569
left_only      31
right_only      0
Name: count, dtype: int64

### Loans with no member: how many, how many members, how much money

In [36]:
no_member = merged[merged["_merge"] == "left_only"]
print("loans:", len(no_member))
print("different members:", no_member["member_id"].nunique())
print("late fees:", no_member["late_fee"].sum())

loans: 31
different members: 6
late fees: 347.5


### Try an inner merge

In [37]:
inner = loans.merge(members, on="member_id", how="inner")
print(inner.shape)
print("late_fee total:", inner["late_fee"].sum())

(569, 11)
late_fee total: 7140.0


**Answer:** After drop_duplicates members has (144, 3). Now the merge works and gives (600, 12), and the late_fee total is 7487.5 like before.
31 loans are left_only. They belong to 6 members who are not in the members table, and they have 347.5 kr in late fees.
An inner merge gives (569, 11) and 7140.0. It just drops those 31 loans without telling me, so the total is wrong.

## Task 8 : Use the merged table

### Give the loans with no member the branch "No member record"

In [38]:
merged["branch"] = merged["branch"].fillna("No member record")
merged["branch"].value_counts()

branch
Central             144
South               130
North               126
East                 72
Harbour              51
Unknown              46
No member record     31
Name: count, dtype: int64

**Answer:** Loans per branch: Central 144, South 130, North 126, East 72, Harbour 51, Unknown 46 and No member record 31.
Central has the most loans.
Unknown and No member record are not the same. Unknown is a member we have, but we don't know their branch.
No member record is a loan where the member is not in the members table at all.